# **E-commerce Data Cleaning, Preprocessing & Feature Engineering**

## **🚀 Data Cleaning Pipeline**
### **1. Library Imports**
The following essential libraries are imported to support data manipulation, numerical computation, and preprocessing throughout this pipeline. `numpy` provides efficient array operations and mathematical functions, while `pandas` serves as the primary framework for loading, transforming, and analyzing structured tabular data.

In [5]:
import pandas as pd
import numpy as np

### **1. Loading Datasets**
Six datasets are loaded into the environment, each representing a distinct dimension of the business's operations. Together, they form the foundation for all subsequent analysis covering customer demographics, product performance, marketing engagement, purchase journeys, geographic distribution, and customer feedback.

In [6]:
customers = pd.read_csv(r"C:\Users\Anubhav\OneDrive\Desktop\Marketing Project\Customers.csv")
reviews = pd.read_csv(r"C:\Users\Anubhav\OneDrive\Desktop\Marketing Project\Customer_reviews.csv")
engagement = pd.read_csv(r"C:\Users\Anubhav\OneDrive\Desktop\Marketing Project\Engagement.csv")
products = pd.read_csv(r"C:\Users\Anubhav\OneDrive\Desktop\Marketing Project\Products.csv")
geography = pd.read_csv(r"C:\Users\Anubhav\OneDrive\Desktop\Marketing Project\Geography.csv")
journey = pd.read_csv(r"C:\Users\Anubhav\OneDrive\Desktop\Marketing Project\Journey.csv")

## 3. Holistic Dataset Summary Function

A reusable diagnostic function `check()` is defined to provide a comprehensive overview of any given DataFrame in a single call. Rather than running multiple individual inspection commands, this function consolidates table name, shape, data type information, null value counts per column, and duplicate row count into one structured output — enabling rapid quality assessment across all datasets.

In [7]:
def check(df, name):
    print(f"\n{name}")
    print(df.shape)
    print(df.info())
    print(df.isnull().sum())
    print(df.duplicated().sum())

## 4. Customers Dataset

### 4.1 Initial Quality Assessment

The `check()` function is applied to the Customers dataset to validate its integrity before any transformations are performed. The dataset contains **100 rows and 6 columns**, all data types are appropriate for their respective fields, and no null values or duplicate records are detected — confirming the dataset is clean and ready for enrichment.

In [8]:
check(customers, "customers")


customers
(100, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Customer_ID   100 non-null    int64 
 1   CustomerName  100 non-null    object
 2   Email         100 non-null    object
 3   Gender        100 non-null    object
 4   Age           100 non-null    int64 
 5   Geography_ID  100 non-null    int64 
dtypes: int64(3), object(3)
memory usage: 4.8+ KB
None
Customer_ID     0
CustomerName    0
Email           0
Gender          0
Age             0
Geography_ID    0
dtype: int64
0


### 4.2 Merging Customers with Geography

To enrich the customer records with geographic context, the Customers dataset is merged with the Geography dataset using `Geography_ID` as the common key. A left join is performed to ensure all customer records are retained, even if a corresponding geography entry is not found.

In [9]:
customers_geo = customers.merge(
    geography,
    on="Geography_ID",
    how="left"
)

### 4.3 Dropping Redundant Columns

Following the merge, the `Geography_ID` and `Email` columns are removed from the dataset. `Geography_ID` is now redundant as the geographic details have been incorporated directly, and `Email` does not contribute any analytical value to demographic or behavioral analysis. Dropping these columns reduces noise and keeps the dataset lean.

In [10]:
customers_geo = customers_geo.drop(columns=["Geography_ID", "Email"])

### 4.4 Engineering the Age Group Feature

A new categorical feature `Age_Group` is derived from the existing `Age` column to enable demographic segmentation. Customers are bucketed into five meaningful age bands that align with standard market segmentation conventions, allowing for group-level analysis of behavior, preferences, and conversion patterns.

In [ ]:
bins = [18, 25, 35, 50, 65, 75]
labels = ["18-25", "26-35", "36-50", "51-65", "65+"]

customers_geo["age_group"] = pd.cut(
    customers_geo["Age"],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)

## 5. Engagement Dataset

### 5.1 Initial Quality Assessment

The `check()` function is applied to the Engagement dataset. The dataset contains **4,623 rows and 8 columns**. While most data types are appropriate, the `EngagementDate` column is stored as a string object and requires conversion to a proper datetime format. No null values or duplicates are identified.

In [19]:
check(engagement, "engagement")


engagement
(4623, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4623 entries, 0 to 4622
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Engagement_ID        4623 non-null   int64 
 1   Content_ID           4623 non-null   int64 
 2   ContentType          4623 non-null   object
 3   Likes                4623 non-null   int64 
 4   EngagementDate       4623 non-null   object
 5   CampaignID           4623 non-null   int64 
 6   Product_ID           4623 non-null   int64 
 7   ViewsClicksCombined  4623 non-null   object
dtypes: int64(5), object(3)
memory usage: 289.1+ KB
None
Engagement_ID          0
Content_ID             0
ContentType            0
Likes                  0
EngagementDate         0
CampaignID             0
Product_ID             0
ViewsClicksCombined    0
dtype: int64
0


### 5.2 Splitting the Combined Views & Clicks Column

The `ViewsClicksCombined` column stores both views and clicks as a concatenated string (e.g., `"1500-200"`). This column is split on the delimiter to extract two independent numeric fields — `Views` and `Clicks` — enabling them to be analyzed and aggregated separately.

In [20]:
engagement[['Views', 'Clicks']] = engagement['ViewsClicksCombined'].str.split('-', expand=True)

### 5.3 Dropping the Combined Column

Once the `Views` and `Clicks` values have been successfully extracted into their own columns, the original `ViewsClicksCombined` column is dropped as it is no longer needed and would otherwise introduce redundancy into the dataset.

In [21]:
engagement = engagement.drop(columns=['ViewsClicksCombined'])

### 5.4 Converting EngagementDate to Datetime

The `EngagementDate` column is converted from its current string representation to a proper `datetime` data type. This transformation is essential for enabling time-based filtering, grouping by month or year, and plotting temporal engagement trends.

In [23]:
engagement['EngagementDate'] = pd.to_datetime(engagement['EngagementDate'], errors='coerce')

### 5.5 Converting Views and Clicks to Numeric

The `Views` and `Clicks` columns — now separated from the combined string — are explicitly cast to numeric data types. Any values that cannot be coerced are set to `NaN` using `errors='coerce'`, ensuring the dataset remains robust against malformed entries.

In [24]:
numeric_eng = ['Views', 'Clicks']

for col in numeric_eng:
    engagement[col] = pd.to_numeric(engagement[col], errors='coerce')

### 5.6 Filling Null Values with Zero

Any null values introduced during the numeric conversion step — or otherwise present in the Views and Clicks columns — are filled with `0`. This approach is appropriate here, as a missing value in this context most likely indicates the absence of activity rather than an unknown value.

In [26]:
engagement = engagement.fillna(0)

### 5.7 Standardizing ContentType to Proper Case

The `ContentType` column is standardized to title case (e.g., `"blog"` → `"Blog"`) to ensure consistency across all categorical values. Inconsistent casing can cause the same content type to be treated as multiple distinct categories during grouping and aggregation.

In [30]:
engagement['ContentType'] = engagement['ContentType'].str.title()

### 5.8 Correcting the Social Media Label

A specific inconsistency is addressed by replacing the improperly formatted value `"Socialmedia"` with the correctly spaced label `"Social Media"`. This ensures uniformity with how this content type is represented across other datasets and in the final visualizations.

In [33]:
engagement['ContentType'] = engagement['ContentType'].astype(str).str.replace('Socialmedia', 'Social Media')

## 6. Journey Dataset

### 6.1 Initial Quality Assessment

The `check()` function is applied to the Journey dataset. The dataset contains **4,011 rows and 7 columns**. The `VisitDate` column requires datetime conversion. Notably, **613 null values** are present in the `Duration` column; however, upon inspection these nulls are exclusively associated with `Drop-Off` actions, meaning customers did not complete the step and therefore no duration was recorded. These nulls are intentional and are left as-is to preserve analytical integrity. Additionally, **71 duplicate rows** are identified and will be removed.

In [35]:
check(journey, "journey")


journey
(4011, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4011 entries, 0 to 4010
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Journey_ID   4011 non-null   int64  
 1   Customer_ID  4011 non-null   int64  
 2   Product_ID   4011 non-null   int64  
 3   VisitDate    4011 non-null   object 
 4   Stage        4011 non-null   object 
 5   Action       4011 non-null   object 
 6   Duration     3398 non-null   float64
dtypes: float64(1), int64(3), object(3)
memory usage: 219.5+ KB
None
Journey_ID       0
Customer_ID      0
Product_ID       0
VisitDate        0
Stage            0
Action           0
Duration       613
dtype: int64
71


### 6.2 Converting VisitDate to Datetime

The `VisitDate` column is converted from a string object to a `datetime` data type to enable time-series analysis, monthly aggregations, and accurate temporal joins with other datasets.

In [36]:
journey['VisitDate'] = pd.to_datetime(journey['VisitDate'], errors='coerce')

### 6.3 Removing Duplicate Records

The 71 identified duplicate rows are removed to ensure that each customer journey event is represented exactly once. Retaining duplicates would inflate funnel stage counts and distort conversion rate calculations.

In [37]:
journey = journey.drop_duplicates()

## 7. Products Dataset

### 7.1 Initial Quality Assessment

The `check()` function is applied to the Products dataset. The dataset contains **20 rows and 4 columns**, all data types are correctly assigned, and no null values or duplicate entries are found — confirming the dataset is clean and ready for feature engineering.

In [41]:
check(products, "products")


products
(20, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Product_ID   20 non-null     int64  
 1   ProductName  20 non-null     object 
 2   Category     20 non-null     object 
 3   Price        20 non-null     float64
dtypes: float64(1), int64(1), object(2)
memory usage: 772.0+ bytes
None
Product_ID     0
ProductName    0
Category       0
Price          0
dtype: int64
0


### 7.2 Engineering the Price Category Feature

A new categorical feature `Price_Category` is created by segmenting the continuous `Price` column into three tiers: **Low**, **Medium**, and **High**. This binning approach enables price-level analysis of conversion rates, customer preferences, and marketing performance — as observed in the Power BI dashboard where medium and high price categories outperformed low-priced products in conversion.

In [42]:
products["price_category"] = pd.cut(
    products["Price"],
    bins=[0, 50, 200, float("inf")],
    labels=["Low", "Medium", "High"],
    right=False
)

## 8. Reviews Dataset

### 8.1 Initial Quality Assessment

The `check()` function is applied to the Reviews dataset. The dataset contains **1,363 rows and 6 columns**. All data types are appropriate with the exception of `Review_Date`, which is stored as a string and requires conversion to datetime. No null values or duplicate records are detected.

In [13]:
check(reviews, "reviews")


reviews
(1363, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1363 entries, 0 to 1362
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Review_ID    1363 non-null   int64 
 1   Customer_ID  1363 non-null   int64 
 2   Product_ID   1363 non-null   int64 
 3   Review_Date  1363 non-null   object
 4   Ratings      1363 non-null   int64 
 5   ReviewText   1363 non-null   object
dtypes: int64(4), object(2)
memory usage: 64.0+ KB
None
Review_ID      0
Customer_ID    0
Product_ID     0
Review_Date    0
Ratings        0
ReviewText     0
dtype: int64
0


### 8.2 Converting Review_Date to Datetime

The `Review_Date` column is converted from a string object to a `datetime` data type, enabling monthly and yearly trend analysis of customer sentiment over time — a key requirement for the sentiment tracking visualizations in the dashboard.

In [14]:
reviews['Review_Date'] = pd.to_datetime(reviews['Review_Date'], errors='coerce')

### 8.3 Normalizing Whitespace in ReviewText

The `ReviewText` column is found to contain inconsistent double spacing between words. All occurrences of double spaces are replaced with single spaces to ensure clean, uniform text input for the downstream natural language processing and sentiment analysis steps.

In [16]:
reviews['ReviewText'] = reviews['ReviewText'].astype(str).str.replace('  ', ' ')

## 9. Sentiment Analysis on Review Text & Ratings

### 9.1 Importing NLTK

The Natural Language Toolkit (`nltk`) library is imported to provide access to the text processing utilities and pre-trained lexicons required for rule-based sentiment analysis.

In [58]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

### 9.2 Downloading the VADER Lexicon

The **VADER (Valence Aware Dictionary and sEntiment Reasoner)** lexicon is downloaded from the NLTK data repository. VADER is a pre-trained sentiment analysis tool specifically designed for short social-media-style texts and customer reviews, making it well-suited for this use case.

In [44]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Anubhav\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

### 9.3 Initializing the VADER Sentiment Analyzer

An instance of the `SentimentIntensityAnalyzer` is initialized. This object will be used throughout the analysis to compute sentiment scores for each review text in the dataset.

In [45]:
sia = SentimentIntensityAnalyzer()

### 9.4 Defining the Sentiment Score Function

A function `calculate_sentiment()` is defined to compute the **compound sentiment score** for a given piece of text using VADER. The compound score is a normalized value ranging from **-1** (most negative) to **+1** (most positive), providing a single continuous measure of overall sentiment that can be used for ranking and categorization.

In [46]:
def calculate_sentiment(review):
    sentiment = sia.polarity_scores(review)
    return sentiment['compound']

### 9.5 Defining the Sentiment Categorization Function

A function `categorize_sentiment()` is defined to classify each review into a meaningful sentiment category by combining both the **VADER compound score** and the **star rating** provided by the customer. This dual-signal approach improves classification accuracy, as text sentiment and numeric ratings do not always align perfectly.

In [47]:
def categorize_sentiment(score, Ratings):
    if score > 0.05:  
        if Ratings >= 4:
            return 'Positive'  
        elif Ratings == 3:
            return 'Mixed Positive' 
        else:
            return 'Mixed Negative' 
    elif score < -0.05: 
        if Ratings <= 2:
            return 'Negative'  
        elif Ratings == 3:
            return 'Mixed Negative'  
        else:
            return 'Mixed Positive' 
    else:  
        if Ratings >= 4:
            return 'Positive'  
        elif Ratings <= 2:
            return 'Negative'  
        else:
            return 'Neutral' 

### 9.6 Defining the Sentiment Bucketing Function

A function `bucket_sentiment()` is defined to group compound scores into discrete, human-readable score ranges. This bucketing supports histogram-style analysis of how sentiment is distributed across the review corpus, independent of the star rating.

In [48]:
def sentiment_bucket(score):
    if score >= 0.5:
        return '0.5 to 1.0'  
    elif 0.0 <= score < 0.5:
        return '0.0 to 0.49'  
    elif -0.5 <= score < 0.0:
        return '-0.49 to 0.0'  
    else:
        return '-1.0 to -0.5' 

### 9.7 Applying Sentiment Scoring

The `calculate_sentiment()` function is applied across the entire `ReviewText` column to compute a compound sentiment score for every review. The results are stored in a new `SentimentScore` column.

In [49]:
reviews['SentimentScore'] = reviews['ReviewText'].apply(calculate_sentiment)

### 9.8 Applying Sentiment Categorization

The `categorize_sentiment()` function is applied row-wise using both the `SentimentScore` and `Ratings` columns to assign each review a final sentiment category: **Positive**, **Negative**, **Mixed Positive**, **Mixed Negative**, or **Neutral**. These categories align directly with the sentiment breakdown visualized in the Power BI dashboard.

In [50]:
reviews['SentimentCategory'] = reviews.apply(
    lambda row: categorize_sentiment(row['SentimentScore'], row['Ratings']), axis=1)

### 9.9 Applying Sentiment Bucketing

The `bucket_sentiment()` function is applied to the `Sentiment_Score` column to assign each review to a defined score range bucket. This enables distribution analysis of raw sentiment strength across the review dataset.

In [51]:
reviews['SentimentBucket'] = reviews['SentimentScore'].apply(sentiment_bucket)

### 9.10 Previewing Sentiment Results

The first few rows of the enriched Reviews DataFrame are displayed to verify that the sentiment scores, categories, and buckets have been correctly computed and appended. This acts as a final sanity check before export.

In [52]:
reviews.head()

,Review_ID,Customer_ID,Product_ID,Review_Date,Ratings,ReviewText,SentimentScore,SentimentCategory,SentimentBucket
0,1,77,18,2023-12-23,3,"Average experience, nothing special.",-0.3089,Mixed Negative,-0.49 to 0.0
1,2,80,19,2024-12-25,5,The quality is top-notch.,0.0000,Positive,0.0 to 0.49
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.,0.0000,Positive,0.0 to 0.49
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper.",0.2382,Mixed Positive,0.0 to 0.49
4,5,64,2,2023-07-16,3,"Average experience, nothing special.",-0.3089,Mixed Negative,-0.49 to 0.0


### 9.12 Exporting Cleaned Datasets to CSV

All five cleaned and enriched DataFrames are exported to CSV files for downstream consumption — including Power BI reporting, further modeling, or archival purposes. Each file is exported without the pandas index to maintain clean, importable files.

In [53]:
reviews.to_csv(r"C:\Users\Anubhav\OneDrive\Desktop\Marketing Project\reviews_sentiment.csv", index=False)

In [54]:
products.to_csv(r"C:\Users\Anubhav\OneDrive\Desktop\Marketing Project\products.csv", index=False)

In [55]:
engagement.to_csv(r"C:\Users\Anubhav\OneDrive\Desktop\Marketing Project\engagement.csv", index=False)

In [56]:
journey.to_csv(r"C:\Users\Anubhav\OneDrive\Desktop\Marketing Project\journey.csv", index=False)

In [57]:
customers_geo.to_csv(r"C:\Users\Anubhav\OneDrive\Desktop\Marketing Project\customers.csv", index=False)